# Pixel-Perfect: Content-Aware Super-Resolution Demo

This notebook demonstrates the usage of the **Pixel-Perfect** model (based on ESRGAN) for upscaling retro game assets. 

### Project Highlights:
1. **Nearest-Neighbor Training Pairs**: Maintains blocky pixel art geometry.
2. **Edge-Aware Sharpness Loss**: Eliminates "oil painting" artifacts.
3. **Palette-Constraint Loss**: Prevents color drifting.

In [ ]:
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import sys
import os

# Add src to path
sys.path.append(os.path.abspath('../src'))
from models.esrgan import RRDBNet

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load the Model

We load the generator model (RRDBNet) and move it to the evaluation mode.

In [ ]:
# Initialize model (8 RRDB blocks used for Pixel-Perfect Milestone 2)
model = RRDBNet(in_nc=3, out_nc=3, nf=64, nb=8, gc=32).to(device)

# Load weights (placeholder path)
weights_path = '../models/checkpoints/generator_epoch_20.pth'
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print("Weights loaded successfully!")
else:
    print(f"WARNING: Weights not found at {weights_path}. Running with random initialization.")

model.eval()

## 2. Preprocess Input

Pixel art assets are typically very small. The model expects a 32x32 input for 4x upscaling to 128x128.

In [ ]:
def preprocess_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img = img.resize((32, 32), Image.NEAREST)
    img_np = np.array(img).astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(np.transpose(img_np[:, :, [2, 1, 0]], (2, 0, 1))).unsqueeze(0).to(device)
    return img_tensor, img

# Load a sample sprite
sample_path = '../mario_clone/resources/graphics/mario_bros.png' # Or any other sprite
try:
    input_tensor, original_img = preprocess_image(sample_path)
except FileNotFoundError:
    print("Sample image not found. Please provide a path to a pixel art sprite.")

## 3. Run Inference

We pass the low-res tensor through the generator.

In [ ]:
with torch.no_grad():
    output = model(input_tensor).squeeze().cpu().clamp(0, 1).numpy()

output = np.transpose(output[[2, 1, 0], :, :], (1, 2, 0))
output_img = (output * 255.0).round().astype(np.uint8)
output_img = Image.fromarray(output_img)

## 4. Visualize Results

Comparison between the Original (1x), Bicubic Upscale (Baseline), and Pixel-Perfect (Ours).

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title("Input (32x32)")
plt.imshow(original_img.resize((32,32), Image.NEAREST))
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title("Bicubic Baseline (128x128)")
plt.imshow(original_img.resize((128, 128), Image.BICUBIC))
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title("Pixel-Perfect (128x128)")
plt.imshow(output_img)
plt.axis('off')

plt.show()